# Lab 39: The Familiarity Glitch

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-39.ipynb)

**What you will do:** record your own postcard or photo recognition errors, see why this Lab has no single published number to match against, and watch a tiny overfitted classifier make the same kind of confident-but-wrong "I recognise this" mistake that déjà vu makes.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 39 you (or a friend) looked through a set of postcards or photos, waited five minutes,
then judged a mixed deck — the original items plus some new ones — as "old" or "new". Some of
the new items feel eerily familiar, especially when they share features (similar
architecture, colours, or composition) with items you actually saw. The familiar feeling is
genuine; the judgement it produces is not.

## Record your results

Tally your recognition test. Split the new items into two kinds: ones you deliberately (or
accidentally) chose because they resemble an original item, and ones that are clearly
different from anything in the original set. This split is what lets you test whether
surface similarity, rather than true exposure, drives the false alarms.

In [ ]:
# example data: replace with your own recognition-test tally
category_n = {"old (seen before)": 30, "new: similar to an old item": 3, "new: clearly different": 2}
category_said_old = {"old (seen before)": 24, "new: similar to an old item": 2, "new: clearly different": 0}  # example data

results = pd.DataFrame({"shown": category_n, 'judged "old"': category_said_old})
results['proportion "old"'] = (results['judged "old"'] / results["shown"]).round(2)
display(results)

results['proportion "old"'].plot.barh(color=["steelblue", "firebrick", "grey"], figsize=(6, 2.6))
plt.xlim(0, 1); plt.xlabel('proportion judged "old"'); plt.gca().invert_yaxis(); plt.show()

## Compare

Lab 39 does not report a numeric false-alarm rate for this exact recognition task — unlike
the DRM lists in Lab 38, there is no published percentage to match here. The one
quantitative figure the text does give measures something different: déjà vu itself has an
estimated lifetime *prevalence* of about two-thirds of people across studies (Brown, 2004),
meaning most people report having the experience at some point — not that two-thirds of new
items get misjudged as old. What we can test is the qualitative prediction of the dual-process
account: familiarity should misfire most often when a new item shares surface features with
something you actually saw, not simply when it is unfamiliar.

In [ ]:
similar_rate = results.loc["new: similar to an old item", 'proportion "old"']
different_rate = results.loc["new: clearly different", 'proportion "old"']
print(f"False alarms on similar-looking new items:   {similar_rate:.0%}")
print(f"False alarms on clearly different new items: {different_rate:.0%}")
if similar_rate > different_rate:
    print("As the dual-process account predicts: surface similarity, not true recognition, drove the false alarms.")
else:
    print("No gap here -- with only a handful of new items, a couple of individual errors can go either way.")
print()
print("Note: the book gives no task-specific benchmark percentage for this recognition test.")
print("The only published number in this Lab is the lifetime prevalence of the deja vu EXPERIENCE")
print("itself (about two-thirds of people, Brown 2004) -- a different measure from this task's")
print("false-alarm rate, reported above for context rather than as a figure to match.")

## The AI side

The book's AI Connection names a **false-positive match in similarity search**, rather than
overfitting in the strict training sense, as the closest computational relative of déjà vu: a
system reports "I have seen this before" because an input resembles something in its index,
even when the resemblance is only superficial. Below, two classifiers are trained on the same
tiny, noisy dataset. One (`k=1`) is a nearest-neighbour model that simply copies the label of
the single closest training example it has ever seen — the AI equivalent of judging by "this
looks just like that one thing", with no way to tell superficial closeness from a genuine
match. The other (`k=11`) averages over eleven neighbours, which smooths out noise at the
cost of a lower score on the training data itself.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = make_moons(n_samples=50, noise=0.3, random_state=9)  # a small, noisy two-class dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=9)

overfit = KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train)    # "memorises" every training point
generalises = KNeighborsClassifier(n_neighbors=11).fit(X_train, y_train)  # averages over many neighbours

ai = pd.DataFrame({
    "train accuracy": [overfit.score(X_train, y_train), generalises.score(X_train, y_train)],
    "test accuracy":  [overfit.score(X_test, y_test), generalises.score(X_test, y_test)],
}, index=["overfit (k=1, 'perfect memory')", "regularised (k=11, generalises)"])
display(ai)

pred = overfit.predict(X_test)
wrong = pred != y_test
if wrong.any():
    conf = overfit.predict_proba(X_test[wrong]).max(axis=1)
    print(f"On the {wrong.sum()} of {len(y_test)} test points k=1 got wrong, "
          f"its average confidence was {conf.mean():.0%} -- confidently wrong, not uncertain.")
else:
    print("k=1 happened to get every test point right this run -- try a different random_state above.")

Notice that the overfit (k=1) model is not just wrong on some test points — when it is
wrong, it is exactly as confident as when it is right, because a 1-nearest-neighbour model
has no way to express doubt: it always copies the single closest training example's label
outright. This is the machine parallel of déjà vu: a familiarity signal that fires as
strongly for a coincidental resemblance as for a real match, with no separate signal to flag
the difference. Real recognition memory is more forgiving than k=1 — your brain can be more
or less sure of something — but the underlying failure is the same: judging by superficial
closeness to one stored example rather than by the deeper structure of the category.

## Questions to think about

1. In your record data, was the false-alarm rate higher for postcards you chose because they looked similar to ones you had seen, compared with ones that were clearly different? What does this tell you about whether familiarity tracks true prior exposure or mere surface resemblance?
2. Look at the k=1 model's confidence on the test points it got wrong in the ai_lab cell. Why can a 1-nearest-neighbour model never express doubt, and how is that similar to, or different from, the vivid "click" of recognition described in the Mooney-image version of the experiment (Step 4)?
3. The regularised (k=11) model scores lower on training accuracy than the overfit (k=1) model, but higher on test accuracy. Which of the two models better describes a healthy recognition-memory system, and which better describes déjà vu?
4. The text distinguishes familiarity (perirhinal cortex) from recollection (hippocampus). If, for every "old" judgement, you had also asked your friend to say *where* in the deck they thought they had seen the item, how would a confident "yes, familiar" answer with no specific source map onto the ai_lab's nearest-neighbour demonstration?

## Challenge

Try the Mooney-image version (Step 4): find or create some two-tone Mooney images, view the
plain photographs they were derived from, then look at the Mooney versions again and note how
suddenly recognisable — and how familiar — they feel.

In [ ]:
# Example data: replace with your own Mooney-image session.
mooney_total = 6                      # example data: how many Mooney images you tested
recognised_before_reveal = 1          # example data: how many you could identify before seeing the originals
recognised_after_reveal = 6           # example data: how many of the same set you recognised after seeing the originals once

print(f"Recognised before seeing the originals: {recognised_before_reveal}/{mooney_total}")
print(f"Recognised after seeing the originals once: {recognised_after_reveal}/{mooney_total}")
print("The jump between these two numbers is itself a memory effect: a single brief exposure to the")
print("clear photograph is often enough to make the degraded Mooney image feel obvious afterwards --")
print("and that sense of obviousness can feel exactly like a rush of familiarity.")
print("The values above are example data -- replace them with your own session's results.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-39.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")